## Write a solution to find the rank of the scores. The ranking should be calculated according to the following rules:

## The scores should be ranked from the highest to the lowest.
If there is a tie between two scores, both should have the same ranking.
After a tie, the next ranking number should be the next consecutive integer value. In other words, there should be no holes between ranks.
Return the result table ordered by score in descending order.

The result format is in the following example.

 

Example 1:

Input: 
Scores table:
+----+-------+
| id | score |
+----+-------+
| 1  | 3.50  |
| 2  | 3.65  |
| 3  | 4.00  |
| 4  | 3.85  |
| 5  | 4.00  |
| 6  | 3.65  |
+----+-------+
Output: 
+-------+------+
| score | rank |
+-------+------+
| 4.00  | 1    |
| 4.00  | 1    |
| 3.85  | 2    |
| 3.65  | 3    |
| 3.65  | 3    |
| 3.50  | 4    |
+-------+------+

## Taking pandas schema and converting to Spark dataFrame


In [1]:
import pandas as pd
data = [[1, 3.5], [2, 3.65], [3, 4.0], [4, 3.85], [5, 4.0], [6, 3.65]]
scores = pd.DataFrame(data, columns=['id', 'score']).astype({'id':'Int64', 'score':'Float64'}) 

In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("rankScore").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 16:49:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df1=spark.createDataFrame(scores)
df1.show()

+---+-----+
| id|score|
+---+-----+
|  1|  3.5|
|  2| 3.65|
|  3|  4.0|
|  4| 3.85|
|  5|  4.0|
|  6| 3.65|
+---+-----+



In [25]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,col,dense_rank
windowSpec=Window.orderBy(col("score").desc())
df2=df1.withColumn("rank",dense_rank().over(windowSpec))
df2.show()

26/06/14 17:20:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 17:20:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 17:20:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+-----+----+
| id|score|rank|
+---+-----+----+
|  3|  4.0|   1|
|  5|  4.0|   1|
|  4| 3.85|   2|
|  2| 3.65|   3|
|  6| 3.65|   3|
|  1|  3.5|   4|
+---+-----+----+



## creating temp view and than findimng the rank Score

In [28]:
df1.createOrReplaceTempView("Score_rank")
sqlDF=spark.sql(
    """
    select id,score,dense_rank() over(order by score desc) as rank from Score_rank"""
)


In [ ]:
sqlDF.show()

26/06/14 17:30:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 17:30:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/14 17:30:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+-----+----+
| id|score|rank|
+---+-----+----+
|  3|  4.0|   1|
|  5|  4.0|   1|
|  4| 3.85|   2|
|  2| 3.65|   3|
|  6| 3.65|   3|
|  1|  3.5|   4|
+---+-----+----+



26/06/17 05:50:02 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 902735 ms exceeds timeout 120000 ms
26/06/17 05:50:02 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/17 05:50:08 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o